# SGLang Serving Runner (Colab GPU)

Run this notebook on a GPU runtime to launch an SGLang OpenAI-compatible endpoint and export a public URL for local benchmarking.

In [1]:
import os, re, sys, time, subprocess
from urllib import request

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sglang', 'pyngrok'])
print('Installed: sglang, pyngrok')

Installed: sglang, pyngrok


In [5]:
SGLANG_MODEL = os.environ.get('SGLANG_MODEL', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0')
SGLANG_PORT = int(os.environ.get('SGLANG_PORT', '30000'))
SGLANG_TP = int(os.environ.get('SGLANG_TP', '1'))
SGLANG_LOG_PATH = '/tmp/sglang_server.log'

if 'sglang_proc' in globals() and sglang_proc is not None and sglang_proc.poll() is None:
    sglang_proc.terminate(); time.sleep(2)
if 'sglang_cfd' in globals() and sglang_cfd is not None and sglang_cfd.poll() is None:
    sglang_cfd.terminate(); time.sleep(1)

subprocess.run(['bash', '-lc', 'pkill -f sglang.launch_server || true'], check=False)
time.sleep(1)

cmd = [sys.executable, '-m', 'sglang.launch_server', '--model-path', SGLANG_MODEL, '--host', '0.0.0.0', '--port', str(SGLANG_PORT), '--tp', str(SGLANG_TP)]
print('Starting:', ' '.join(cmd))
sglang_proc = subprocess.Popen(cmd, stdout=open(SGLANG_LOG_PATH, 'w'), stderr=subprocess.STDOUT, text=True)
time.sleep(25)

PUBLIC_BASE_URL = None
try:
    from pyngrok import ngrok
    if os.environ.get('NGROK_AUTHTOKEN'): ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
    PUBLIC_BASE_URL = ngrok.connect(SGLANG_PORT, 'http').public_url.rstrip('/')
except Exception:
    subprocess.run(['bash', '-lc', "which cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)"], check=True)
    sglang_cfd = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{SGLANG_PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    pat = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')
    t0 = time.time()
    while time.time() - t0 < 35:
        line = sglang_cfd.stdout.readline() if sglang_cfd.stdout else ''
        m = pat.search(line) if line else None
        if m: PUBLIC_BASE_URL = m.group(0); break

print('PUBLIC_BASE_URL=', PUBLIC_BASE_URL)
for _ in range(30):
    try:
        with request.urlopen(request.Request(PUBLIC_BASE_URL + '/v1/models', method='GET'), timeout=8) as resp:
            print('Health HTTP', resp.status)
            print(resp.read(300).decode('utf-8', 'replace'))
            break
    except Exception as exc:
        print('Health retry:', type(exc).__name__, exc)
        time.sleep(2)

print('SGLang running:', sglang_proc.poll() is None)

Starting: /usr/bin/python3 -m sglang.launch_server --model-path TinyLlama/TinyLlama-1.1B-Chat-v1.0 --host 0.0.0.0 --port 30000 --tp 1


ERROR:pyngrok.process.ngrok:t=2026-03-25T18:07:19+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PUBLIC_BASE_URL= https://img-garbage-liver-exception.trycloudflare.com
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URLError <urlopen error [Errno -2] Name or service not known>
Health retry: URL

In [4]:
print('Local machine commands:')
print(
    'uv run python scripts/cloud/write_remote_config.py '
    f'--backend sglang --base-url {PUBLIC_BASE_URL} --r5 '
    f"--streaming-model '{SGLANG_MODEL}' --agentic-model '{SGLANG_MODEL}' "
    '--output configs/live_sglang_remote_r5.json'
)
print('uv run python scripts/probe_live_endpoints.py --config configs/live_sglang_remote_r5.json')
print('uv run mws-bench sweep --config configs/live_sglang_remote_r5.json --output results/live_sglang_remote_sweep_r5.csv --trace-output-dir results/traces/live_sglang_remote_r5')

Local machine commands:
uv run python scripts/cloud/write_remote_config.py --backend sglang --base-url https://worked-meetings-pierre-lawsuit.trycloudflare.com --r5 --streaming-model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' --agentic-model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' --output configs/live_sglang_remote_r5.json
uv run python scripts/probe_live_endpoints.py --config configs/live_sglang_remote_r5.json
uv run mws-bench sweep --config configs/live_sglang_remote_r5.json --output results/live_sglang_remote_sweep_r5.csv --trace-output-dir results/traces/live_sglang_remote_r5
